In [ ]:
import sys
from rdflib import Graph, URIRef, BNode, Literal
from rdflib.namespace import RDF, RDFS
from pathlib import Path

root = Path.cwd().parent
sys.path.insert(0, str(root))
from config import DATA_DIR

In [ ]:
g = Graph()
g.parse(str(DATA_DIR / "processed" / "supplychain_kg.ttl"), format="turtle")

In [ ]:
def split_uri(uri):
    uri = str(uri)
    if "#" in uri:
        return uri.rsplit("#", 1)[-1]
    return uri.rstrip("/").rsplit("/", 1)[-1]

def format_literal(value):
    text = str(value).strip()
    if not text:
        return ""
    if isinstance(value, Literal):
        if value.language:
            return f"{text}@{value.language}"
        if value.datatype:
            return f"{text}^^{split_uri(value.datatype)}"
    return f"{text}" if " " in text or not text.isidentifier() else text

def node_to_text(node):
    if isinstance(node, URIRef):
        uri = str(node)
        parts = uri.rstrip("/").split("/")
        token = split_uri(node)
        if len(parts) >= 2 and parts[-1].isdigit():
            return f"{parts[-2].capitalize()} {parts[-1]}"
        return token
    if isinstance(node, BNode):
        return f"blank node {str(node)}"
    if isinstance(node, Literal):
        return format_literal(node)
    return str(node)

def predicate_key(predicate):
    if predicate == RDF.type:
        return "type"
    if predicate == RDFS.subClassOf:
        return "subClassOf"
    key = split_uri(predicate)
    mapping = {
        "leadTimeDays": "lead time",
        "reliabilityScore": "reliability score",
        "onTimeDeliveryRate": "on-time delivery rate",
        "qualityIssueCount": "quality issue count",
        "riskRating": "risk rating",
        "capacityTier": "capacity tier",
        "performanceCategory": "performance category",
        "exposureScore": "exposure score",
        "supplierType": "supplier type",
        "contractType": "contract type",
    }
    return mapping.get(key, key)

def article(word):
    word = str(word).strip()
    return "an" if word[:1].lower() in "aeiou" else "a"

def capitalize_sentence(text):
    text = text.strip()
    return text[:1].upper() + text[1:] if text else text

def make_sentence(subject, predicate, obj):
    s = node_to_text(subject)
    p = predicate_key(predicate)
    o = node_to_text(obj)

    templates = {
        "type": lambda s, o: f"{s} is {article(o)} {o}.",
        "subClassOf": lambda s, o: f"{s} is a subclass of {o}.",
        "partOf": lambda s, o: f"{s} is part of {o}.",
        "locatedIn": lambda s, o: f"{s} is located in {o}.",
        "worksFor": lambda s, o: f"{s} works for {o}.",
        "hasSupplier": lambda s, o: f"{s} has supplier {o}.",
        "hasProduct": lambda s, o: f"{s} has product {o}.",
        "produces": lambda s, o: f"{s} produces {o}.",
        "contains": lambda s, o: f"{s} contains {o}.",
        "relatedTo": lambda s, o: f"{s} is related to {o}.",
        "label": lambda s, o: f"{s} is labeled {o}.",
        "name": lambda s, o: f"{s} is named {o}.",
        "manufacturedBy": lambda s, o: f"{s} is manufactured by {o}.",
        "shippedTo": lambda s, o: f"{s} is shipped to {o}.",
        "storedAt": lambda s, o: f"{s} is stored at {o}.",
        "suppliesTo": lambda s, o: f"{s} supplies to {o}.",
        "lead time": lambda s, o: f"{s} has lead time {o} days.",
        "reliability score": lambda s, o: f"{s} has reliability score {o}.",
        "on-time delivery rate": lambda s, o: f"{s} has an on-time delivery rate of {o}.",
        "quality issue count": lambda s, o: f"{s} has {o} quality issue count.",
        "risk rating": lambda s, o: f"{s} has a risk rating of {o}.",
        "capacity tier": lambda s, o: f"{s} has capacity tier {o}.",
        "performance category": lambda s, o: f"{s} has performance category {o}.",
        "exposure score": lambda s, o: f"{s} has exposure score {o}.",
        "supplier type": lambda s, o: f"{s} is a {o} supplier.",
        "contract type": lambda s, o: f"{s} has contract type {o}.",
    }

    if p in templates:
        return capitalize_sentence(templates[p](s, o))

    if isinstance(obj, Literal):
        return capitalize_sentence(f"{s} has {p} value {o}.")

    return capitalize_sentence(f"{s} has {p} {o}.")

def kg_to_text(g, deduplicate=True, sort_output=True):
    text_chunks = []
    seen = set()

    triples = g.triples((None, None, None))
    if sort_output:
        triples = sorted(triples, key=lambda t: (str(t[0]), str(t[1]), str(t[2])))

    for s, p, o in triples:
        if not isinstance(s, (URIRef, BNode)):
            continue

        sentence = make_sentence(s, p, o)
        if deduplicate:
            if sentence in seen:
                continue
            seen.add(sentence)

        text_chunks.append(sentence)

    return text_chunks


In [ ]:
text_chunks = kg_to_text(g)

output_path = DATA_DIR / 'processed' / 'supplychain_kg_text.txt'

with open(str(output_path), "w", encoding="utf-8") as f:
    f.write("\n".join(text_chunks) + "\n")

# Use relative path for output message to avoid exposing absolute paths
relative_path = output_path.relative_to(root.parent)
print(f"Exported {len(text_chunks)} knowledge graph text chunks to {relative_path}.")